[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/39.%20The%20Channel%20Dredging%20%26%20Deepening%20Investment%20Problem/P39-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 39. The Channel Dredging and Deepening Investment Problem

## Tier 1: Multi-Objective Programming Formulation

### Goal and workflow

This notebook formulates a channel-dredging investment problem as a mixed-integer linear program (MILP). It uses a weighted-sum objective to balance discounted financial value, channel accessibility, and environmental impact. Binary decision variables represent dredging choices and the channel configuration used in each year. The model is solved with PuLP and the CBC solver; a restricted enumeration method is available as a fallback for this small example.

The workflow is to define the example data, calculate the cost and benefit components, build the mathematical model, solve it, inspect the investment schedule and performance measures, and examine how objective weights and the annual budget affect the solution.

### Example data and scope

The illustrative Melbourne Port Channel Expansion scenario has three segments, a current depth of 11.6 m, target depths of 12, 13, and 14 m, target widths of 200 and 250 m, a five-year planning horizon, and an 8% annual discount rate. Monetary amounts are expressed in millions of Australian dollars (million AUD).

The model treats the channel as having one operating depth-and-width configuration in any year. That configuration is feasible only when every segment has been dredged by that year to at least the selected depth and width. A segment may be dredged at most once. Benefits and maintenance costs accrue in the dredging year and each later year in the planning horizon. The model does not represent uncertainty, construction duration within a year, traffic assignment, or segment-specific operating configurations. Cost, benefit, and environmental-impact relationships are illustrative inputs rather than engineering estimates. The soil-specific dredging coefficients are illustrative cost coefficients applied to depth increase and segment length; they should not be interpreted as validated engineering unit rates.

### Weighted-sum objective

The weighted sum combines three criteria. Financial net present value (NPV) is measured in million AUD, accessibility is a discounted sum of normalized annual depth scores, and environmental impact is a discounted cost in million AUD. Since these criteria have different units, the weights define a scoring convention; they are not percentages of a single common physical quantity.

Let $S$ be the set of channel segments, $D$ the set of target depths, $W$ the set of target widths, and $T=\{0,1,\ldots,H-1\}$ the planning years. The binary decision $x_{sdwt}$ equals 1 when segment $s$ is dredged to depth $d$ and width $w$ in year $t$. The binary decision $y_{dwt}$ equals 1 when the channel operates at configuration $(d,w)$ in year $t$. Let $C_{sdw}$ be the capital cost of dredging segment $s$ to $(d,w)$, $M_{dw}$ the annual maintenance cost, $B_{dw}$ the annual operating benefit, and $E_{sdw}$ the one-time environmental impact cost. The normalized accessibility score is $A_d=(d-11.6)/(14-11.6)$, and the discount factor in year $t$ is $q_t=(1+r)^{-t}$, where $r$ is the annual discount rate.

The objective is

$$\max\quad \lambda_N\left[\sum_{t\in T}q_t\sum_{d\in D}\sum_{w\in W}(B_{dw}-M_{dw})y_{dwt}-\sum_{s\in S}\sum_{d\in D}\sum_{w\in W}\sum_{t\in T}q_t C_{sdw}x_{sdwt}\right]+\lambda_A\sum_{t\in T}q_t\sum_{d\in D}\sum_{w\in W}A_dy_{dwt}-\lambda_E\sum_{s\in S}\sum_{d\in D}\sum_{w\in W}\sum_{t\in T}q_t E_{sdw}x_{sdwt}.$$

Here, $\lambda_N$, $\lambda_A$, and $\lambda_E$ are the financial, accessibility, and environmental weights, respectively. Financial NPV includes benefits, maintenance, and capital costs. Environmental cost is a separate weighted criterion rather than being counted a second time inside financial NPV. The code assembles these components in `create_optimization_model`.

### Constraints and interpretation

The constraints express the operational rules of the investment plan. Each segment can receive at most one upgrade over the full horizon. At most one upgraded operating configuration can be selected in a year. A configuration can be selected only if, by that year, every segment has been upgraded to a depth and width at least as large as the configuration. Annual capital spending cannot exceed the budget, and total segment-depth improvement must be at least 1 m. The model may select no upgraded operating configuration in a year; this represents the baseline configuration and earns no modeled benefit or accessibility score.

Using $B$ for the annual capital budget, the constraints are

$$\sum_{d\in D}\sum_{w\in W}\sum_{t\in T}x_{sdwt}\leq 1 \qquad \forall s\in S,$$

$$\sum_{d\in D}\sum_{w\in W}y_{dwt}\leq 1 \qquad \forall t\in T,$$

$$y_{dwt}\leq \sum_{\substack{d'\in D:d'\geq d\\w'\in W:w'\geq w}}\sum_{\tau=0}^{t}x_{sd'w'\tau} \qquad \forall s\in S,\ d\in D,\ w\in W,\ t\in T,$$

$$\sum_{s\in S}\sum_{d\in D}\sum_{w\in W}C_{sdw}x_{sdwt}\leq B \qquad \forall t\in T,$$

$$\sum_{s\in S}\sum_{d\in D}\sum_{w\in W}\sum_{t\in T}(d-d_s^{\mathrm{current}})x_{sdwt}\geq 1,$$

$$x_{sdwt}\in\{0,1\},\qquad y_{dwt}\in\{0,1\}.$$

In the completion constraint, $d'$ and $w'$ range over target options that are at least as large as the operating depth $d$ and width $w$, and $\tau$ ranges over the current year and all earlier years. The constraint is imposed for every segment, so one segment's upgrade cannot by itself make the shared channel configuration available. In the minimum-improvement constraint, $d_s^{\mathrm{current}}$ is the current depth of segment $s$. In the code, the corresponding variables are `x` and `y`; the annual budget and improvement constraints are assembled in `create_optimization_model`.

These linear constraints with binary decisions define a MILP. A MILP solver uses exact optimization methods, including branch-and-bound, to search the feasible integer decisions. In the worst case, the number of binary assignments grows exponentially with the number of decision variables, although solver performance depends on the structure of the instance. The restricted enumeration fallback checks a smaller set of schedules and is not a certificate of global optimality for the full MILP. The sensitivity analyses show how the selected solution responds to the specified weights and budget scenarios; they do not establish that any one weighting is universally preferable.

In [ ]:
# Import the libraries used for optimization, data handling, and visualization.
import itertools
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pulp import (
    LpMaximize,
    LpProblem,
    LpVariable,
    PULP_CBC_CMD,
    lpSum,
)
from pulp.constants import LpStatus

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the segments, options, and economic parameters

The data classes keep each input's meaning and units visible. Depth and width are in meters, segment length is in kilometers, the discount rate is a fraction per year, and the annual budget is in million AUD. The planning years used by the model are indexed from 0, so year 0 has discount factor 1.

In [ ]:
@dataclass
class ChannelSegment:
    """Physical and cost characteristics of one channel segment."""

    id: int
    current_depth: float  # meters
    length: float  # kilometers
    soil_type: str  # sand, mixed, or rock


@dataclass
class DredgingOption:
    """Target depth and width for a dredging or operating configuration."""

    depth: float  # meters
    width: float  # meters


@dataclass
class EconomicParameters:
    """Economic inputs for the finite planning horizon."""

    discount_rate: float
    planning_horizon: int  # years
    annual_budget_limit: float  # million AUD


# Illustrative Melbourne Port Channel Expansion data.
segments = [
    ChannelSegment(1, 11.6, 5.2, "sand"),
    ChannelSegment(2, 11.6, 4.8, "mixed"),
    ChannelSegment(3, 11.6, 6.1, "rock"),
]

depth_options = [12.0, 13.0, 14.0]
width_options = [200.0, 250.0]

economic_params = EconomicParameters(
    discount_rate=0.08,
    planning_horizon=5,
    annual_budget_limit=300.0,
)

print("Problem setup:")
print(f"- Number of segments: {len(segments)}")
print(f"- Depth options (m): {depth_options}")
print(f"- Width options (m): {width_options}")
print(f"- Planning horizon: {economic_params.planning_horizon} years")
print(f"- Discount rate: {economic_params.discount_rate:.1%}")
print(f"- Annual budget limit: {economic_params.annual_budget_limit:.1f} million AUD")

## 2. Define the cost and benefit functions

The capital-cost function scales with the depth increase, segment length, soil type, and width factor. The soil-specific coefficients are illustrative coefficients used with depth increase in meters and segment length in kilometers; their resulting values are modeled in million AUD and are not engineering unit-rate estimates. Annual benefit and maintenance functions depend on the operating depth and width. Each function returns a value in million AUD, except that the accessibility score used later is dimensionless. These functions translate the illustrative assumptions into coefficients used by both the optimization model and solution reporting.

In [ ]:
def calculate_dredging_cost(
    segment: ChannelSegment,
    option: DredgingOption,
) -> float:
    """Estimate capital cost in million AUD for one segment upgrade."""
    base_cost_per_meter = {
        "sand": 25.0,
        "mixed": 35.0,
        "rock": 50.0,
    }
    depth_increase = option.depth - segment.current_depth
    width_factor = 1.0 if option.width == 200.0 else 1.25
    return (
        base_cost_per_meter[segment.soil_type]
        * depth_increase
        * segment.length
        * width_factor
    )


def calculate_annual_benefit(option: DredgingOption) -> float:
    """Return annual operating benefit in million AUD."""
    depth_benefits = {
        12.0: 45.0,
        13.0: 75.0,
        14.0: 120.0,
    }
    width_bonus = 1.0 if option.width == 200.0 else 1.15
    return depth_benefits[option.depth] * width_bonus


def calculate_maintenance_cost(option: DredgingOption) -> float:
    """Return annual maintenance cost in million AUD."""
    base_maintenance = 5.0
    depth_factor = option.depth / 11.6
    width_factor = option.width / 200.0
    return base_maintenance * depth_factor * width_factor


def calculate_environmental_cost(
    segment: ChannelSegment,
    option: DredgingOption,
) -> float:
    """Estimate one-time environmental impact cost in million AUD."""
    return (
        (option.depth - segment.current_depth)
        * (option.width / 200.0)
        * 2.0
    )


# A worked coefficient example makes the units and inputs concrete.
test_segment = segments[0]
test_option = DredgingOption(13.0, 200.0)
print("Cost and benefit calculation example:")
print(
    f"Segment {test_segment.id}: {test_segment.current_depth:.1f} m "
    f"to {test_option.depth:.1f} m"
)
print(
    "Capital cost: "
    f"{calculate_dredging_cost(test_segment, test_option):.1f} million AUD"
)
print(
    "Annual benefit: "
    f"{calculate_annual_benefit(test_option):.1f} million AUD"
)
print(
    "Annual maintenance: "
    f"{calculate_maintenance_cost(test_option):.1f} million AUD"
)
print(
    "Environmental impact cost: "
    f"{calculate_environmental_cost(test_segment, test_option):.1f} million AUD"
)

## 3. Build the mixed-integer optimization model

The binary variables encode yes-or-no decisions. The capital-cost terms are attached to dredging decisions, while benefits, maintenance, and accessibility are attached to the single channel operating configuration for each year. The completion constraints link those decisions: an operating depth and width are available only after every segment can support them. This link is essential because one upgraded segment alone cannot make the entire channel usable at that configuration.

The code below implements the weighted objective and constraints stated above. The annual budget is applied to undiscounted capital spending in the year it occurs; discounting is used in the objective and reported present-value metrics.

In [ ]:
def create_optimization_model(
    segments: List[ChannelSegment],
    depth_options: List[float],
    width_options: List[float],
    economic_params: EconomicParameters,
    weights: Tuple[float, float, float] = (0.4, 0.4, 0.2),
) -> LpProblem:
    """Create the weighted-sum MILP for channel dredging decisions."""
    if len(weights) != 3 or any(weight < 0 for weight in weights):
        raise ValueError("Provide three non-negative objective weights.")
    if not math.isclose(sum(weights), 1.0, rel_tol=1e-9, abs_tol=1e-9):
        raise ValueError("Objective weights must sum to 1.")
    if economic_params.planning_horizon <= 0:
        raise ValueError("The planning horizon must be positive.")

    horizon = economic_params.planning_horizon
    model = LpProblem("Channel_Dredging_Investment", LpMaximize)

    # x[s,d,w,t] records a one-time upgrade decision for a segment.
    x = {
        (s, d, w, t): LpVariable(
            f"x_{s}_{d}_{w}_{t}", cat="Binary"
        )
        for s in range(len(segments))
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    }

    # y[d,w,t] records the shared channel operating configuration.
    y = {
        (d, w, t): LpVariable(
            f"y_{d}_{w}_{t}", cat="Binary"
        )
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    }

    capital_cost = lpSum(
        calculate_dredging_cost(segments[s], DredgingOption(d, w))
        * x[(s, d, w, t)]
        / ((1 + economic_params.discount_rate) ** t)
        for s in range(len(segments))
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    )

    environmental_cost = lpSum(
        calculate_environmental_cost(
            segments[s], DredgingOption(d, w)
        )
        * x[(s, d, w, t)]
        / ((1 + economic_params.discount_rate) ** t)
        for s in range(len(segments))
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    )

    operating_npv = lpSum(
        (
            calculate_annual_benefit(DredgingOption(d, w))
            - calculate_maintenance_cost(DredgingOption(d, w))
        )
        * y[(d, w, t)]
        / ((1 + economic_params.discount_rate) ** t)
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    )

    accessibility = lpSum(
        ((d - 11.6) / (14.0 - 11.6))
        * y[(d, w, t)]
        / ((1 + economic_params.discount_rate) ** t)
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    )

    financial_npv = operating_npv - capital_cost
    model += (
        weights[0] * financial_npv
        + weights[1] * accessibility
        - weights[2] * environmental_cost
    )

    # A segment receives at most one upgrade over the entire horizon.
    for s in range(len(segments)):
        model += lpSum(
            x[(s, d, w, t)]
            for d in depth_options
            for w in width_options
            for t in range(horizon)
        ) <= 1

    # Each year has at most one upgraded operating configuration.
    for t in range(horizon):
        model += lpSum(
            y[(d, w, t)] for d in depth_options for w in width_options
        ) <= 1

    # Every segment must already support the selected global configuration.
    for d in depth_options:
        for w in width_options:
            for t in range(horizon):
                for s in range(len(segments)):
                    eligible_completed_upgrades = lpSum(
                        x[(s, upgrade_depth, upgrade_width, tau)]
                        for upgrade_depth in depth_options
                        for upgrade_width in width_options
                        if upgrade_depth >= d and upgrade_width >= w
                        for tau in range(t + 1)
                    )
                    model += y[(d, w, t)] <= eligible_completed_upgrades

    # Capital spending is limited in each calendar year, without discounting.
    for t in range(horizon):
        annual_capital = lpSum(
            calculate_dredging_cost(
                segments[s], DredgingOption(d, w)
            )
            * x[(s, d, w, t)]
            for s in range(len(segments))
            for d in depth_options
            for w in width_options
        )
        model += annual_capital <= economic_params.annual_budget_limit

    # Require at least one meter of total segment-depth improvement.
    total_improvement = lpSum(
        (d - segments[s].current_depth) * x[(s, d, w, t)]
        for s in range(len(segments))
        for d in depth_options
        for w in width_options
        for t in range(horizon)
    )
    model += total_improvement >= 1.0

    return model


print("Optimization model construction function defined.")

## 4. Solve the model and extract decisions

The solver returns values for the binary variables. `extract_solution` converts selected variables into readable schedules and annual configurations. A value above 0.5 identifies a selected binary decision; the threshold safely interprets small numerical tolerances in solver output. If CBC does not report an optimal solution, the solver function calls the restricted enumeration fallback defined in the next section.

In [ ]:
def extract_solution(model: LpProblem) -> Dict:
    """Convert selected model variables into schedules and configurations."""
    solution = {
        "dredging_schedule": [],
        "channel_configurations": [],
    }

    for variable in model.variables():
        if variable.varValue is None or variable.varValue <= 0.5:
            continue

        parts = variable.name.split("_")
        if variable.name.startswith("x_"):
            solution["dredging_schedule"].append(
                {
                    "segment": int(parts[1]),
                    "depth": float(parts[2]),
                    "width": float(parts[3]),
                    "year": int(parts[4]),
                }
            )
        elif variable.name.startswith("y_"):
            solution["channel_configurations"].append(
                {
                    "depth": float(parts[1]),
                    "width": float(parts[2]),
                    "year": int(parts[3]),
                }
            )

    solution["dredging_schedule"].sort(
        key=lambda item: (item["year"], item["segment"])
    )
    solution["channel_configurations"].sort(key=lambda item: item["year"])
    return solution


def solve_optimization_model(
    segments: List[ChannelSegment],
    depth_options: List[float],
    width_options: List[float],
    economic_params: EconomicParameters,
    weights: Tuple[float, float, float] = (0.4, 0.4, 0.2),
) -> Tuple[Dict, Dict]:
    """Solve the MILP and report solution metrics."""
    print(f"Solving with weights: {weights}")
    model = create_optimization_model(
        segments, depth_options, width_options, economic_params, weights
    )
    solver = PULP_CBC_CMD(msg=False, timeLimit=60)
    model.solve(solver)

    if LpStatus[model.status] != "Optimal":
        print("CBC did not report an optimal solution; using restricted enumeration.")
        return solve_by_enumeration(
            segments, depth_options, width_options, economic_params, weights
        )

    solution = extract_solution(model)
    metrics = calculate_solution_metrics(
        solution, segments, economic_params
    )
    metrics["weighted_objective"] = calculate_weighted_objective(
        metrics, weights
    )
    return solution, metrics


print("Solution extraction and solver functions defined.")

## 5. Restricted enumeration fallback

Enumeration evaluates a finite list of candidate investment choices rather than relying on the MILP solver. For each segment, this fallback considers either no upgrade or a 13 m upgrade at a specified width and year. It checks the annual budget and minimum-improvement requirement, then selects the best feasible operating configuration in each year from those supported by all segments. This restricted candidate set makes the fallback practical for the example, but it does not cover every decision available to the MILP and therefore cannot certify the MILP optimum.

In [ ]:
def check_budget_constraints(
    schedule: List[Dict],
    segments: List[ChannelSegment],
    economic_params: EconomicParameters,
) -> bool:
    """Check annual capital spending against the budget limit."""
    annual_costs = [0.0] * economic_params.planning_horizon
    for item in schedule:
        option = DredgingOption(item["depth"], item["width"])
        annual_costs[item["year"]] += calculate_dredging_cost(
            segments[item["segment"]], option
        )
    return all(
        cost <= economic_params.annual_budget_limit + 1e-9
        for cost in annual_costs
    )


def generate_configurations(
    schedule: List[Dict],
    segments: List[ChannelSegment],
    depth_options: List[float],
    width_options: List[float],
    economic_params: EconomicParameters,
    weights: Tuple[float, float, float],
) -> List[Dict]:
    """Choose the best supported operating configuration in each year."""
    configurations = []
    for year in range(economic_params.planning_horizon):
        available_upgrades = {
            item["segment"]: item
            for item in schedule
            if item["year"] <= year
        }
        candidates = []
        for depth in depth_options:
            for width in width_options:
                supported_by_all_segments = all(
                    segment_index in available_upgrades
                    and available_upgrades[segment_index]["depth"] >= depth
                    and available_upgrades[segment_index]["width"] >= width
                    for segment_index in range(len(segments))
                )
                if not supported_by_all_segments:
                    continue

                option = DredgingOption(depth, width)
                discount_factor = 1 / (
                    (1 + economic_params.discount_rate) ** year
                )
                annual_score = discount_factor * (
                    weights[0]
                    * (
                        calculate_annual_benefit(option)
                        - calculate_maintenance_cost(option)
                    )
                    + weights[1] * (depth - 11.6) / (14.0 - 11.6)
                )
                candidates.append((annual_score, depth, width))

        if candidates and max(candidates)[0] > 0:
            _, depth, width = max(candidates)
            configurations.append(
                {"depth": depth, "width": width, "year": year}
            )
        else:
            configurations.append(
                {"depth": 11.6, "width": 200.0, "year": year}
            )
    return configurations


def calculate_weighted_objective(
    metrics: Dict,
    weights: Tuple[float, float, float],
) -> float:
    """Combine reported objective components using the specified weights."""
    return (
        weights[0] * metrics["npv"]
        + weights[1] * metrics["accessibility_score"]
        - weights[2] * metrics["environmental_cost"]
    )


def solve_by_enumeration(
    segments: List[ChannelSegment],
    depth_options: List[float],
    width_options: List[float],
    economic_params: EconomicParameters,
    weights: Tuple[float, float, float],
) -> Tuple[Dict, Dict]:
    """Search the fallback's restricted set of dredging schedules."""
    print("Using restricted enumeration fallback.")
    possible_years = [
        year
        for year in (0, 2, 4)
        if year < economic_params.planning_horizon
    ]
    fallback_widths = [width for width in width_options if width in (200.0, 250.0)]
    choices = [None]
    choices.extend(
        {"depth": 13.0, "width": width, "year": year}
        for width in fallback_widths
        for year in possible_years
    )

    best_solution: Optional[Dict] = None
    best_objective = -float("inf")
    for selected_choices in itertools.product(choices, repeat=len(segments)):
        schedule = [
            {"segment": s, **choice}
            for s, choice in enumerate(selected_choices)
            if choice is not None
        ]
        total_improvement = sum(
            item["depth"] - segments[item["segment"]].current_depth
            for item in schedule
        )
        if total_improvement < 1.0 - 1e-9:
            continue
        if not check_budget_constraints(schedule, segments, economic_params):
            continue

        configurations = generate_configurations(
            schedule,
            segments,
            depth_options,
            width_options,
            economic_params,
            weights,
        )
        candidate_solution = {
            "dredging_schedule": schedule,
            "channel_configurations": configurations,
        }
        candidate_metrics = calculate_solution_metrics(
            candidate_solution, segments, economic_params
        )
        objective = calculate_weighted_objective(candidate_metrics, weights)
        if objective > best_objective + 1e-9:
            best_objective = objective
            best_solution = candidate_solution

    if best_solution is None:
        raise RuntimeError("The restricted enumeration set has no feasible schedule.")

    metrics = calculate_solution_metrics(
        best_solution, segments, economic_params
    )
    metrics["weighted_objective"] = calculate_weighted_objective(
        metrics, weights
    )
    return best_solution, metrics


print("Enumeration and supporting functions defined.")

## 6. Calculate consistent solution metrics

Metrics are recomputed from the selected schedule and annual configurations using the same timing and discounting conventions as the model. Financial NPV includes discounted benefits, maintenance, and capital costs. Environmental impact is reported separately. The accessibility metric is the discounted sum of annual normalized accessibility scores, matching the accessibility term in the optimization objective; it is not a percentage of a single-year score.

In [ ]:
def calculate_solution_metrics(
    solution: Dict,
    segments: List[ChannelSegment],
    economic_params: EconomicParameters,
) -> Dict:
    """Calculate discounted financial, environmental, and access measures."""
    if not solution:
        return {}

    total_capital_cost = 0.0
    total_environmental_cost = 0.0
    for item in solution["dredging_schedule"]:
        segment = segments[item["segment"]]
        option = DredgingOption(item["depth"], item["width"])
        discount_factor = 1 / (
            (1 + economic_params.discount_rate) ** item["year"]
        )
        total_capital_cost += (
            calculate_dredging_cost(segment, option) * discount_factor
        )
        total_environmental_cost += (
            calculate_environmental_cost(segment, option) * discount_factor
        )

    total_benefits = 0.0
    total_maintenance = 0.0
    accessibility_score = 0.0
    for configuration in solution["channel_configurations"]:
        depth = configuration["depth"]
        width = configuration["width"]
        year = configuration["year"]
        if depth <= 11.6:
            continue

        option = DredgingOption(depth, width)
        discount_factor = 1 / (
            (1 + economic_params.discount_rate) ** year
        )
        total_benefits += (
            calculate_annual_benefit(option) * discount_factor
        )
        total_maintenance += (
            calculate_maintenance_cost(option) * discount_factor
        )
        accessibility_score += (
            (depth - 11.6) / (14.0 - 11.6) * discount_factor
        )

    npv = total_benefits - total_capital_cost - total_maintenance
    return {
        "total_capital_cost": total_capital_cost,
        "total_benefits": total_benefits,
        "total_maintenance": total_maintenance,
        "environmental_cost": total_environmental_cost,
        "npv": npv,
        "accessibility_score": accessibility_score,
        "benefit_cost_ratio": (
            total_benefits / total_capital_cost
            if total_capital_cost > 0
            else 0.0
        ),
    }


print("Solution metric function defined.")

## 7. Solve and inspect the main scenario

The default weights are $(0.4,0.4,0.2)$ for financial NPV, accessibility, and environmental impact. The output reports the selected upgrade decisions and the associated present-value metrics. A solution is an investment plan under the stated assumptions and weighted objective, not a forecast or a universal recommendation.

In [ ]:
weights = (0.4, 0.4, 0.2)
solution, metrics = solve_optimization_model(
    segments,
    depth_options,
    width_options,
    economic_params,
    weights,
)

print("\nSelected dredging schedule:")
for item in solution["dredging_schedule"]:
    segment = segments[item["segment"]]
    print(
        f"Year {item['year']}: Segment {segment.id} ({segment.soil_type}) "
        f"from {segment.current_depth:.1f} m to {item['depth']:.1f} m, "
        f"width {item['width']:.0f} m"
    )

print("\nOperating configurations by year:")
for item in solution["channel_configurations"]:
    if item["depth"] <= 11.6:
        print(f"Year {item['year']}: baseline configuration (11.6 m, 200 m)")
    else:
        print(
            f"Year {item['year']}: {item['depth']:.1f} m depth, "
            f"{item['width']:.0f} m width"
        )

print("\nPerformance metrics:")
for key, value in metrics.items():
    if key == "accessibility_score":
        print(f"{key.replace('_', ' ').title()}: {value:.3f} discounted score-years")
    elif key == "benefit_cost_ratio":
        print(f"{key.replace('_', ' ').title()}: {value:.3f}")
    else:
        print(f"{key.replace('_', ' ').title()}: {value:.2f} million AUD")

assert len(solution["dredging_schedule"]) <= len(segments)
assert math.isclose(
    metrics["npv"],
    metrics["total_benefits"]
    - metrics["total_capital_cost"]
    - metrics["total_maintenance"],
    rel_tol=1e-9,
    abs_tol=1e-9,
)

## 8. Sensitivity to objective weights

A weighted-sum model can select different plans when the relative importance of its criteria changes. The scenarios below retain the same data, constraints, solver, and optimization formulation while changing only the weights. Compare the resulting financial NPV, discounted accessibility score, and capital spending. Since accessibility and monetary measures have different units, interpret the weight scenarios as alternative scoring choices rather than directly comparable percentages.

In [ ]:
weight_scenarios = [
    (0.6, 0.3, 0.1),
    (0.3, 0.6, 0.1),
    (0.4, 0.4, 0.2),
    (0.5, 0.3, 0.2),
]

weight_results = []
for scenario_weights in weight_scenarios:
    scenario_solution, scenario_metrics = solve_optimization_model(
        segments,
        depth_options,
        width_options,
        economic_params,
        scenario_weights,
    )
    weight_results.append(
        {
            "weights": scenario_weights,
            "npv": scenario_metrics["npv"],
            "accessibility": scenario_metrics["accessibility_score"],
            "capital_cost": scenario_metrics["total_capital_cost"],
            "environmental_cost": scenario_metrics["environmental_cost"],
        }
    )

comparison_df = pd.DataFrame(
    [
        {
            "Weights (NPV, access, environment)": str(item["weights"]),
            "NPV (million AUD)": item["npv"],
            "Discounted accessibility": item["accessibility"],
            "Capital cost (million AUD)": item["capital_cost"],
            "Environmental cost (million AUD)": item["environmental_cost"],
        }
        for item in weight_results
    ]
)
print(comparison_df.to_string(index=False, float_format="%.2f"))

## 9. Visualize the plan and weight sensitivity

The schedule chart displays the year in which each segment is dredged. The cost-benefit chart reports discounted monetary components, and the sensitivity chart uses separate vertical axes because NPV and accessibility have different units. The final panel shows the selected shared operating depth across the planning years; a baseline year is shown at the current depth.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Channel Dredging Investment Analysis",
    fontsize=16,
    fontweight="bold",
)

# Dredging schedule: horizontal bars show the selected investment year.
ax1 = axes[0, 0]
colors = sns.color_palette("husl", n_colors=len(segments))
for item in solution["dredging_schedule"]:
    segment_index = item["segment"]
    year = item["year"]
    ax1.barh(
        segment_index,
        1,
        left=year,
        height=0.6,
        color=colors[segment_index],
        alpha=0.85,
        label=f"Segment {segment_index + 1}",
    )
    ax1.text(
        year + 0.5,
        segment_index,
        f"{item['depth']:.0f} m",
        ha="center",
        va="center",
        fontweight="bold",
    )
ax1.set_xlabel("Planning year")
ax1.set_ylabel("Channel segment")
ax1.set_title("Dredging schedule")
ax1.set_yticks(range(len(segments)))
ax1.set_yticklabels([f"Segment {segment.id}" for segment in segments])
ax1.set_xlim(0, economic_params.planning_horizon)
ax1.grid(True, axis="x", alpha=0.3)
handles, labels = ax1.get_legend_handles_labels()
if handles:
    unique_labels = dict(zip(labels, handles))
    ax1.legend(unique_labels.values(), unique_labels.keys(), title="Segment")

# Present-value financial components.
ax2 = axes[0, 1]
cost_benefit_labels = [
    "Capital cost",
    "Maintenance cost",
    "Operating benefits",
]
cost_benefit_values = [
    metrics["total_capital_cost"],
    metrics["total_maintenance"],
    metrics["total_benefits"],
]
bars = ax2.bar(
    cost_benefit_labels,
    cost_benefit_values,
    color=["firebrick", "darkorange", "seagreen"],
    alpha=0.75,
)
ax2.set_ylabel("Present value (million AUD)")
ax2.set_title("Discounted financial components")
ax2.tick_params(axis="x", rotation=25)
for bar, value in zip(bars, cost_benefit_values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}",
        ha="center",
        va="bottom",
    )

# NPV and accessibility use separate axes because their units differ.
ax3 = axes[1, 0]
scenario_labels = [
    f"{item['weights'][0]:.1f}, {item['weights'][1]:.1f}, "
    f"{item['weights'][2]:.1f}"
    for item in weight_results
]
x_positions = np.arange(len(scenario_labels))
bar_width = 0.36
ax3_twin = ax3.twinx()
npv_bars = ax3.bar(
    x_positions - bar_width / 2,
    [item["npv"] for item in weight_results],
    bar_width,
    color="royalblue",
    alpha=0.75,
    label="Financial NPV",
)
access_bars = ax3_twin.bar(
    x_positions + bar_width / 2,
    [item["accessibility"] for item in weight_results],
    bar_width,
    color="seagreen",
    alpha=0.75,
    label="Discounted accessibility",
)
ax3.set_xlabel("Weights (NPV, accessibility, environment)")
ax3.set_ylabel("Financial NPV (million AUD)", color="royalblue")
ax3_twin.set_ylabel("Discounted accessibility (score-years)", color="seagreen")
ax3.set_title("Objective-weight sensitivity")
ax3.set_xticks(x_positions)
ax3.set_xticklabels(scenario_labels, rotation=20)
ax3.legend([npv_bars, access_bars], ["Financial NPV", "Discounted accessibility"])

# Shared operating depth over the planning horizon.
ax4 = axes[1, 1]
years = [item["year"] for item in solution["channel_configurations"]]
depths = [item["depth"] for item in solution["channel_configurations"]]
ax4.plot(years, depths, marker="o", linewidth=2.5, color="navy")
ax4.fill_between(years, [11.6] * len(years), depths, alpha=0.25, color="skyblue")
ax4.axhline(
    y=11.6,
    color="firebrick",
    linestyle="--",
    label="Current depth",
)
ax4.set_xlabel("Planning year")
ax4.set_ylabel("Operating depth (m)")
ax4.set_title("Channel operating depth")
ax4.set_xticks(range(economic_params.planning_horizon))
ax4.set_ylim(11, 15)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Sensitivity to the annual budget

This experiment changes the annual capital budget while keeping the discount rate, horizon, data, and objective weights fixed. Each scenario is solved with the same MILP formulation and CBC solver. If the solver does not report an optimal solution for a budget scenario, the table marks its metrics as unavailable rather than interpreting solver failure as a zero-value investment plan.

In [ ]:
budget_scenarios = [200, 250, 300, 350, 400]
budget_results = []

for budget in budget_scenarios:
    scenario_params = EconomicParameters(
        discount_rate=economic_params.discount_rate,
        planning_horizon=economic_params.planning_horizon,
        annual_budget_limit=float(budget),
    )
    model = create_optimization_model(
        segments,
        depth_options,
        width_options,
        scenario_params,
        weights,
    )
    model.solve(PULP_CBC_CMD(msg=False, timeLimit=30))

    if LpStatus[model.status] == "Optimal":
        scenario_solution = extract_solution(model)
        scenario_metrics = calculate_solution_metrics(
            scenario_solution, segments, scenario_params
        )
        budget_results.append(
            {
                "budget": budget,
                "npv": scenario_metrics["npv"],
                "projects_completed": len(
                    scenario_solution["dredging_schedule"]
                ),
                "status": "Optimal",
            }
        )
    else:
        budget_results.append(
            {
                "budget": budget,
                "npv": np.nan,
                "projects_completed": np.nan,
                "status": LpStatus[model.status],
            }
        )

budget_df = pd.DataFrame(budget_results)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(
    budget_df["budget"],
    budget_df["npv"],
    marker="o",
    linewidth=2,
    color="royalblue",
)
axes[0].set_xlabel("Annual budget limit (million AUD)")
axes[0].set_ylabel("Financial NPV (million AUD)")
axes[0].set_title("Financial NPV by annual budget")
axes[0].grid(True, alpha=0.3)

axes[1].bar(
    budget_df["budget"],
    budget_df["projects_completed"],
    width=25,
    color="seagreen",
    alpha=0.75,
)
axes[1].set_xlabel("Annual budget limit (million AUD)")
axes[1].set_ylabel("Number of segment upgrades")
axes[1].set_title("Selected upgrades by annual budget")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(budget_df.to_string(index=False, float_format="%.2f"))

## Summary and interpretation

The notebook constructs a weighted-sum MILP for phased channel dredging. Binary upgrade decisions determine when and how each segment is dredged; binary operating decisions determine the shared channel configuration available in each year. The model enforces one upgrade at most per segment, annual capital budgets, a minimum total depth improvement, and the requirement that all segments support an operating configuration before it is selected.

Interpret the solver output in the context of the stated assumptions and weights. A higher weight on one criterion can alter the plan, while the annual budget can restrict the timing and feasibility of investments. The sensitivity tables and plots provide the evidence for those comparisons when the notebook is run; no particular outcome is guaranteed independently of the input data and solver result.

The model's strengths are its explicit decision variables, transparent constraints, and optimization-based solution for the defined finite instance. Its limitations include reliance on illustrative parameter values, a simplified environmental-cost estimate, deterministic inputs, and the assumption of a single shared operating configuration. MILP solution effort can grow substantially as the numbers of segments, options, and planning years increase. Any practical application would require validated engineering and economic data and careful review of the model scope.